# Utah — Title 31A (Insurance Code) → `data/utah/ins_codes/*.md`

Utah’s **Insurance Code** is **Title 31A** of the **Utah Code**. On **Justia**, the crawl root is **[`/codes/utah/title-31a/`](https://law.justia.com/codes/utah/title-31a/)**; sections often sit under **`chapter-…/part-…/section-…`**. The path segment after **`section-`** is only the **section tail** (e.g. **`101`**), so this notebook builds a **composite label** **`{chapter}-{section-tail}`** (e.g. **`1-101`**) for filenames, sorting, and display (**`Utah Code Ann. § 31A-1-101`**).

**Cloudflare** often blocks plain **`httpx`**; this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`**.

**Discovery:** BFS from the Title 31A index, following only paths under **`/codes/utah/title-31a/`** that are **not** section pages, skipping **`appendix`**, **`chronological-history`**, and **`title-notes`**; collect every section link (many **parts** → a **large** section count; discovery can take several minutes).

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`UT_sec_<slug>.md`** with **`<slug>`** normalized from the composite label (e.g. `1-101` → `UT_sec_1_101.md`; `4-105-5` → `UT_sec_4_105_5.md`).

Config: **MAX_SECTIONS** (**0** = all), **MAX_DISCOVERY_PAGES** (**0** = no cap). **REUSE_DISCOVERED_URLS** skips discovery when **`_ut_title31a_section_urls.txt`** exists.

Run with the **`ins_ipynb/`** directory as cwd. Then **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/utah/title-31a"
TITLE_INDEX = f"{BASE}{PATH_PREFIX}/"

OUT_DIR = Path("data") / "utah" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_ut_title31a_section_urls.txt"
REUSE_DISCOVERED_URLS = True

SKIP_PATH_SUBSTR = ("appendix", "chronological-history", "title-notes")


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def skip_path(p: str) -> bool:
    low = p.lower()
    return any(s in low for s in SKIP_PATH_SUBSTR)


def discover_section_urls() -> list[str]:
    """BFS title-31a index + chapter/part pages; collect section URLs."""
    from collections import deque

    start = TITLE_INDEX
    seen: set[str] = set()
    in_q: set[str] = {path_key(start).lower()}
    q: deque[str] = deque([start])
    sections: set[str] = set()
    fetches = 0
    while q:
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            break
        url = q.popleft()
        pk = path_key(url).lower()
        in_q.discard(pk)
        if pk in seen:
            continue
        if "/section-" in pk.lower():
            continue
        seen.add(pk)
        html = curl_get(url)
        fetches += 1
        if fetches % 25 == 0:
            print(f"… discovery fetch {fetches}, queue={len(q)}, sections={len(sections)}")
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = urljoin(url, a["href"])
            p = path_key(absu).lower()
            if not p.startswith(PATH_PREFIX):
                continue
            if skip_path(p):
                continue
            if "/section-" in p.lower():
                sections.add(BASE + p + "/")
            else:
                if p in seen or p in in_q:
                    continue
                in_q.add(p)
                q.append(BASE + p + "/")
    return sorted(sections, key=lambda u: label_sort_key(section_label_from_url(u)))


def section_label_from_url(url: str) -> str:
    """Composite chapter + section slug (URL `section-` tail is not unique across chapters)."""
    path = path_key(url)
    low = path.lower()
    if "/section-" not in low:
        raise ValueError(f"not a section URL: {url!r}")
    sec_slug = path.rsplit("/section-", 1)[1]
    m = re.search(r"/chapter-([^/]+)/", path, re.I)
    if m:
        return f"{m.group(1)}-{sec_slug}"
    return sec_slug


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_display_citation(label: str) -> str:
    """1-101 → Utah Code Ann. § 31A-1-101"""
    return f"Utah Code Ann. § 31A-{label}"


def label_to_filename(label: str) -> str:
    safe = re.sub(r"[^0-9a-zA-Z]+", "_", label).strip("_").lower()
    return f"UT_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and ("Utah Code" in s or "U.C.A." in s):
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("Utah Code Ann."):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_title_31a() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs under Title 31A")
        all_urls = sorted(found, key=lambda u: label_sort_key(section_label_from_url(u)))
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        disp = label_to_display_citation(label)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t or f"Utah Code {disp}"
                md = (
                    f"# {title}\n\n"
                    f"**Utah Code — Title 31A (Insurance Code)**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [Utah Legislature — Title 31A](https://le.utah.gov/xcode/Title31A/31A.html)\n\n"
                    f"**Section (composite slug):** {label}\n\n"
                    f"**Citation (display):** {disp}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_title_31a()


… discovery fetch 25, queue=79, sections=98
… discovery fetch 50, queue=157, sections=145
… discovery fetch 75, queue=132, sections=312
… discovery fetch 100, queue=107, sections=520
… discovery fetch 125, queue=82, sections=851
… discovery fetch 150, queue=57, sections=1030
… discovery fetch 175, queue=32, sections=1235
… discovery fetch 200, queue=7, sections=1353
Discovered 1381 section URLs under Title 31A
… 200/1381 (wrote=200 skipped=0 failed=0)
… 400/1381 (wrote=400 skipped=0 failed=0)
… 600/1381 (wrote=600 skipped=0 failed=0)
… 800/1381 (wrote=800 skipped=0 failed=0)
… 1000/1381 (wrote=1000 skipped=0 failed=0)
… 1200/1381 (wrote=1200 skipped=0 failed=0)
Done. wrote=1381 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/utah/ins_codes


{'wrote': 1381, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
